# 04 — Locked final evaluation

The design lock from notebook 03 is loaded before the final-test outcomes are evaluated. This notebook fits each selected specification once on all development rows, scores the untouched final test, and generates the committed validation artifacts.

In [1]:
from pathlib import Path
import json
import joblib
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.modeling import fit_locked_models
from src.validation import (metric_row, risk_decile_table, capacity_table, plot_roc_pr,
    plot_calibration, plot_bad_rate_by_decile, plot_cumulative_capture, plot_capacity_tradeoff)

PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "outputs" / "tables"
FIGURES = ROOT / "outputs" / "figures"
MODELS = ROOT / "models"

features = pd.read_pickle(PROCESSED / "borrower_features.pkl")
development = joblib.load(MODELS / "development_result.joblib")
lock = json.loads((MODELS / "design_lock.json").read_text(encoding="utf-8"))
locked = fit_locked_models(features, development)
final = locked["final_test"]
y_final = final["TARGET"].to_numpy()

/Users/yfl/GptWorkspace/projects/credit-risk-default-prediction-python/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/yfl/GptWorkspace/projects/credit-risk-default-prediction-python/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/yfl/GptWorkspace/projects/credit-risk-default-prediction-python/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


/Users/yfl/GptWorkspace/projects/credit-risk-default-prediction-python/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [2]:
metric_rows = []
for model, score in locked["predictions"].items():
    metric_rows.append(metric_row(y_final, score, model, "overall", development.calibration[model]["use_calibration"]))

# Required subgroup checks for age and family status; no legal-fairness claim is made.
age_group = pd.cut(final["APP_AGE_YEARS"], [0, 30, 45, 60, np.inf], right=False, labels=["age_<30", "age_30_44", "age_45_59", "age_60+"])
for model, score in locked["predictions"].items():
    for label in age_group.dropna().unique():
        mask = age_group.eq(label).to_numpy()
        if y_final[mask].sum() >= 25 and (~y_final[mask].astype(bool)).sum() >= 25:
            metric_rows.append(metric_row(y_final[mask], score[mask], model, str(label), development.calibration[model]["use_calibration"]))
    for label in final["APP_FAMILY_STATUS"].fillna("Missing").unique():
        mask = final["APP_FAMILY_STATUS"].fillna("Missing").eq(label).to_numpy()
        if y_final[mask].sum() >= 25 and (~y_final[mask].astype(bool)).sum() >= 25:
            metric_rows.append(metric_row(y_final[mask], score[mask], model, f"family_{label}", development.calibration[model]["use_calibration"]))

final_metrics = pd.DataFrame(metric_rows)
deciles = pd.concat([risk_decile_table(y_final, score, model) for model, score in locked["predictions"].items()], ignore_index=True)
capacity = pd.concat([capacity_table(y_final, score, model) for model, score in locked["predictions"].items()], ignore_index=True)

final_metrics.to_csv(TABLES / "final_test_model_metrics.csv", index=False)
deciles.to_csv(TABLES / "risk_decile_summary.csv", index=False)
capacity.to_csv(TABLES / "capacity_analysis.csv", index=False)
locked["lightgbm_importance"].to_csv(TABLES / "challenger_feature_importance.csv", index=False)

stability = development.logistic_stability.drop(columns=["odds_ratio"], errors="ignore")
coefficients = locked["logistic_coefficients"].merge(stability, on="feature", how="left", validate="one_to_one").sort_values("abs_coefficient_mean", ascending=False)
coefficients.to_csv(TABLES / "logistic_coefficients.csv", index=False)

plot_roc_pr(y_final, locked["predictions"], FIGURES / "roc_pr_model_comparison.png")
plot_calibration(y_final, locked["predictions"], FIGURES / "calibration_comparison.png")
plot_bad_rate_by_decile(deciles, FIGURES / "bad_rate_by_decile.png")
plot_cumulative_capture(deciles, FIGURES / "cumulative_bad_capture.png")
plot_capacity_tradeoff(capacity, FIGURES / "capacity_tradeoff.png")

/Users/yfl/GptWorkspace/projects/credit-risk-default-prediction-python/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/yfl/GptWorkspace/projects/credit-risk-default-prediction-python/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/yfl/GptWorkspace/projects/credit-risk-default-prediction-python/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: invalid value encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/yfl/GptWorkspace/projects/credit-risk-default-prediction-python/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in m

/Users/yfl/GptWorkspace/projects/credit-risk-default-prediction-python/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/yfl/GptWorkspace/projects/credit-risk-default-prediction-python/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/yfl/GptWorkspace/projects/credit-risk-default-prediction-python/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: invalid value encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights


In [3]:
assert len(final) == len(development.split_ids["final_test"])
assert deciles.groupby("model")["borrowers"].sum().eq(len(final)).all()
assert deciles.groupby("model")["bads"].sum().eq(y_final.sum()).all()
assert deciles.groupby("model")["cumulative_bad_capture"].last().pipe(np.allclose, 1.0)
expected = [int(np.ceil(len(final) * x)) for x in [0.05, 0.10, 0.20]]
assert all(group["reviewed_borrowers"].tolist() == expected for _, group in capacity.groupby("model", sort=False))
display(final_metrics[final_metrics["sample"].eq("overall")])
display(capacity)
print(f"Development-selected primary model: {lock['primary_model']}")

,model,sample,calibrated,roc_auc,gini,average_precision,ks,brier,calibration_slope,calibration_intercept,event_rate,rows
0,Logistic,overall,False,0.724815,0.449629,0.202454,0.339166,0.069755,0.998628,-0.000664,0.080728,61503
1,LightGBM,overall,False,0.755276,0.510552,0.237666,0.380747,0.068052,1.028655,0.071372,0.080728,61503


,model,capacity_share,reviewed_borrowers,population_share,captured_bads,observed_event_count,bad_capture_share,event_capture,cumulative_event_capture,reviewed_bad_rate,observed_event_rate,lift_vs_portfolio,non_events_reviewed,reviews_per_captured_event
0,Logistic,0.05,3076,0.050014,876,876,0.176435,0.176435,0.176435,0.284785,0.284785,3.527726,2200,3.511416
1,Logistic,0.10,6151,0.100011,1481,1481,0.298288,0.298288,0.298288,0.240774,0.240774,2.982541,4670,4.153275
2,Logistic,0.20,12301,0.200007,2351,2351,0.473515,0.473515,0.473515,0.191123,0.191123,2.367496,9950,5.232242
3,LightGBM,0.05,3076,0.050014,1017,1017,0.204834,0.204834,0.204834,0.330624,0.330624,4.095545,2059,3.024582
4,LightGBM,0.10,6151,0.100011,1676,1676,0.337563,0.337563,0.337563,0.272476,0.272476,3.375245,4475,3.670048
5,LightGBM,0.20,12301,0.200007,2593,2593,0.522256,0.522256,0.522256,0.210796,0.210796,2.611194,9708,4.743926


Development-selected primary model: LightGBM


The final test is a reproducible random holdout, not an out-of-time sample. Capacity rows describe risk concentration at prespecified review volumes; they are not optimized cutoffs and contain no assumed approval economics. Penalized Logistic coefficients are associative, not causal. Age and family-status subgroup results are diagnostic only and do not establish legal fairness or compliance.